# MobileViT Experimental Pipeline

This notebook evaluates MobileViT on CIFAR-10 using the same workflow as ResNet-18 and EfficientNet-B0: baseline training, pruning, quantization, evaluation, and comparison.

In [6]:
%pip install timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 38.4 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Uninstalling click-8.1.7:
      Successfully uninstalled click-8.1.7
  Attempting uninstall: typer
    Found existing installation: typer 0.26.4
    Uninstalling typer-0.26.4:
      Successfully uninstalled typer-0.26.4
Note: you may need to restart the kernel to use updated packages.


## Environment Verification

This section verifies that all required libraries are installed correctly before beginning the MobileViT experiments. PyTorch is used for model training and evaluation, while timm provides access to the MobileViT architecture.

In [4]:
import torch
print(torch.__version__)

2.5.1


In [7]:
import timm
print(timm.__version__)

1.0.27


## Environment Setup

This section imports the required libraries and configures the computation device for model training and evaluation.

In [7]:
# Import required libraries
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import time
import timm

# Configure computation device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## Data Preparation

This section loads and preprocesses the CIFAR-10 dataset. Images are resized to 224 × 224 pixels to match the input requirements of MobileViT.

In [9]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

Files already downloaded and verified
Files already downloaded and verified


In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training samples: 50000
Testing samples: 10000
Training batches: 1563
Testing batches: 313


## Baseline MobileViT Model

A baseline MobileViT-XXS model is initialized using the timm library. The model is configured for CIFAR-10 classification and trained using the Adam optimizer and cross-entropy loss function.

In [12]:
# Load MobileViT model from timm
mobilevit_model = timm.create_model(
    "mobilevit_xxs",
    pretrained=False,
    num_classes=10
)

# Move model to computation device
mobilevit_model = mobilevit_model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    mobilevit_model.parameters(),
    lr=0.001
)

print("MobileViT model initialized.")

MobileViT model initialized.


## Baseline MobileViT Training

The baseline MobileViT model is trained for five epochs on the CIFAR-10 training dataset. Training loss and accuracy are recorded to evaluate learning performance.

In [14]:
import time
import torch
from torch.utils.data import Subset, DataLoader

# Tiny CPU-friendly run
tiny_train_dataset = Subset(train_dataset, range(500))

train_loader = DataLoader(
    tiny_train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

num_epochs = 1

mobilevit_train_losses = []
mobilevit_train_accuracies = []

start_time = time.time()

for epoch in range(num_epochs):
    mobilevit_model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        predicted = torch.argmax(logits, dim=1)
        total_samples += batch_size
        correct_predictions += (predicted == labels).sum().item()

        print(
            f"Batch [{batch_idx + 1}/{len(train_loader)}] "
            f"Loss: {loss.item():.4f}"
        )

    epoch_loss = running_loss / total_samples
    epoch_accuracy = 100 * correct_predictions / total_samples

    mobilevit_train_losses.append(epoch_loss)
    mobilevit_train_accuracies.append(epoch_accuracy)

end_time = time.time()
mobilevit_training_time = end_time - start_time

print(f"\nTraining Loss: {epoch_loss:.4f}")
print(f"Training Accuracy: {epoch_accuracy:.2f}%")
print(f"Training Time: {mobilevit_training_time:.2f} seconds")

Batch [1/63] Loss: 0.6087
Batch [2/63] Loss: 0.9902
Batch [3/63] Loss: 0.5364
Batch [4/63] Loss: 0.7080
Batch [5/63] Loss: 0.8261
Batch [6/63] Loss: 0.6168
Batch [7/63] Loss: 0.5584
Batch [8/63] Loss: 0.7754
Batch [9/63] Loss: 1.0242
Batch [10/63] Loss: 1.8930
Batch [11/63] Loss: 0.7401
Batch [12/63] Loss: 0.7985
Batch [13/63] Loss: 1.6791
Batch [14/63] Loss: 0.8974
Batch [15/63] Loss: 0.8945
Batch [16/63] Loss: 0.3769
Batch [17/63] Loss: 1.5237
Batch [18/63] Loss: 0.6923
Batch [19/63] Loss: 1.7539
Batch [20/63] Loss: 0.6745
Batch [21/63] Loss: 0.4791
Batch [22/63] Loss: 1.1210
Batch [23/63] Loss: 0.4083
Batch [24/63] Loss: 0.5427
Batch [25/63] Loss: 1.9060
Batch [26/63] Loss: 0.8397
Batch [27/63] Loss: 0.8356
Batch [28/63] Loss: 1.2548
Batch [29/63] Loss: 0.7602
Batch [30/63] Loss: 0.4891
Batch [31/63] Loss: 0.7498
Batch [32/63] Loss: 2.1162
Batch [33/63] Loss: 1.6127
Batch [34/63] Loss: 1.2609
Batch [35/63] Loss: 1.3780
Batch [36/63] Loss: 0.9920
Batch [37/63] Loss: 1.2524
Batch [38/

## Initial MobileViT Feasibility Test

This experiment evaluates whether the MobileViT implementation trains and produces reasonable predictions before scaling to larger datasets.

In [15]:
import torch
from torch.utils.data import Subset, DataLoader

tiny_test_dataset = Subset(test_dataset, range(200))

test_loader = DataLoader(
    tiny_test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

mobilevit_model.eval()

correct_predictions = 0
total_samples = 0

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        predicted = torch.argmax(logits, dim=1)

        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        print(f"Evaluation batch [{batch_idx + 1}/{len(test_loader)}]")

mobilevit_test_accuracy = 100 * correct_predictions / total_samples

print(f"\nBaseline MobileViT Test Accuracy: {mobilevit_test_accuracy:.2f}%")

Evaluation batch [1/25]
Evaluation batch [2/25]
Evaluation batch [3/25]
Evaluation batch [4/25]
Evaluation batch [5/25]
Evaluation batch [6/25]
Evaluation batch [7/25]
Evaluation batch [8/25]
Evaluation batch [9/25]
Evaluation batch [10/25]
Evaluation batch [11/25]
Evaluation batch [12/25]
Evaluation batch [13/25]
Evaluation batch [14/25]
Evaluation batch [15/25]
Evaluation batch [16/25]
Evaluation batch [17/25]
Evaluation batch [18/25]
Evaluation batch [19/25]
Evaluation batch [20/25]
Evaluation batch [21/25]
Evaluation batch [22/25]
Evaluation batch [23/25]
Evaluation batch [24/25]
Evaluation batch [25/25]

Baseline MobileViT Test Accuracy: 58.50%


## Incremental MobileViT Training Under CPU Constraints

### Experiment 1: Medium-Scale CPU Feasibility Test (1,000 Training Images)

This experiment increases the training dataset to 1,000 CIFAR-10 images to evaluate whether MobileViT performance improves while remaining computationally feasible on CPU hardware.

In [18]:
import time
import torch
from torch.utils.data import Subset, DataLoader

# Bigger run than before: 1000 training images
tiny_train_dataset = Subset(train_dataset, range(1000))

train_loader = DataLoader(
    tiny_train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

num_epochs = 1

mobilevit_train_losses = []
mobilevit_train_accuracies = []

start_time = time.time()

for epoch in range(num_epochs):
    mobilevit_model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        predicted = torch.argmax(logits, dim=1)
        total_samples += batch_size
        correct_predictions += (predicted == labels).sum().item()

        if batch_idx % 10 == 0:
            print(
                f"Batch [{batch_idx + 1}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )

    epoch_loss = running_loss / total_samples
    epoch_accuracy = 100 * correct_predictions / total_samples

    mobilevit_train_losses.append(epoch_loss)
    mobilevit_train_accuracies.append(epoch_accuracy)

end_time = time.time()
mobilevit_training_time = end_time - start_time

print(f"\nTraining Loss: {epoch_loss:.4f}")
print(f"Training Accuracy: {epoch_accuracy:.2f}%")
print(f"Training Time: {mobilevit_training_time:.2f} seconds")

Batch [1/125] Loss: 0.5600
Batch [11/125] Loss: 0.9690
Batch [21/125] Loss: 1.3737
Batch [31/125] Loss: 0.9752
Batch [41/125] Loss: 1.3051
Batch [51/125] Loss: 2.3271
Batch [61/125] Loss: 1.2710
Batch [71/125] Loss: 0.8433
Batch [81/125] Loss: 0.4958
Batch [91/125] Loss: 0.6709
Batch [101/125] Loss: 0.5124
Batch [111/125] Loss: 1.1425
Batch [121/125] Loss: 1.2028

Training Loss: 1.0636
Training Accuracy: 64.80%
Training Time: 17.85 seconds


### Experiment 1 Evaluation (500 Test Images)

In [19]:
import torch
from torch.utils.data import Subset, DataLoader

medium_test_dataset = Subset(test_dataset, range(500))

test_loader = DataLoader(
    medium_test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

mobilevit_model.eval()

correct_predictions = 0
total_samples = 0

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        predicted = torch.argmax(logits, dim=1)

        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        if batch_idx % 10 == 0:
            print(f"Evaluation batch [{batch_idx + 1}/{len(test_loader)}]")

mobilevit_test_accuracy = 100 * correct_predictions / total_samples

print(f"\nMobileViT Test Accuracy on 500 test images: {mobilevit_test_accuracy:.2f}%")

Evaluation batch [1/63]
Evaluation batch [11/63]
Evaluation batch [21/63]
Evaluation batch [31/63]
Evaluation batch [41/63]
Evaluation batch [51/63]
Evaluation batch [61/63]

MobileViT Test Accuracy on 500 test images: 63.60%


### Experiment 2: Larger-Scale CPU Feasibility Test (2,000 Training Images)

This experiment further increases the training dataset to 2,000 images to examine the effect of additional training data on MobileViT performance and computational cost.

In [20]:
import time
import torch
from torch.utils.data import Subset, DataLoader

larger_train_dataset = Subset(train_dataset, range(2000))

train_loader = DataLoader(
    larger_train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

num_epochs = 1

mobilevit_train_losses = []
mobilevit_train_accuracies = []

start_time = time.time()

for epoch in range(num_epochs):
    mobilevit_model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        predicted = torch.argmax(logits, dim=1)
        total_samples += batch_size
        correct_predictions += (predicted == labels).sum().item()

        if batch_idx % 20 == 0:
            print(
                f"Batch [{batch_idx + 1}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )

    epoch_loss = running_loss / total_samples
    epoch_accuracy = 100 * correct_predictions / total_samples

    mobilevit_train_losses.append(epoch_loss)
    mobilevit_train_accuracies.append(epoch_accuracy)

end_time = time.time()
mobilevit_training_time = end_time - start_time

print(f"\nTraining Loss: {epoch_loss:.4f}")
print(f"Training Accuracy: {epoch_accuracy:.2f}%")
print(f"Training Time: {mobilevit_training_time:.2f} seconds")

Batch [1/250] Loss: 1.9875
Batch [21/250] Loss: 0.5434
Batch [41/250] Loss: 1.0033
Batch [61/250] Loss: 0.7443
Batch [81/250] Loss: 0.9606
Batch [101/250] Loss: 0.6173
Batch [121/250] Loss: 1.1034
Batch [141/250] Loss: 1.7080
Batch [161/250] Loss: 0.7296
Batch [181/250] Loss: 0.6726
Batch [201/250] Loss: 1.3103
Batch [221/250] Loss: 0.8309
Batch [241/250] Loss: 1.1016

Training Loss: 1.0106
Training Accuracy: 64.05%
Training Time: 35.13 seconds


### Experiment 2 Evaluation (1,000 Test Images)

In [21]:
import torch
from torch.utils.data import Subset, DataLoader

larger_test_dataset = Subset(test_dataset, range(1000))

test_loader = DataLoader(
    larger_test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

mobilevit_model.eval()

correct_predictions = 0
total_samples = 0

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        predicted = torch.argmax(logits, dim=1)

        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        if batch_idx % 20 == 0:
            print(f"Evaluation batch [{batch_idx + 1}/{len(test_loader)}]")

mobilevit_test_accuracy = 100 * correct_predictions / total_samples

print(f"\nMobileViT Test Accuracy on 1000 test images: {mobilevit_test_accuracy:.2f}%")

Evaluation batch [1/125]
Evaluation batch [21/125]
Evaluation batch [41/125]
Evaluation batch [61/125]
Evaluation batch [81/125]
Evaluation batch [101/125]
Evaluation batch [121/125]

MobileViT Test Accuracy on 1000 test images: 63.40%


## Baseline MobileViT Training — Full CIFAR-10 Dataset

The full CIFAR-10 training dataset is used to train the baseline MobileViT model. This serves as the main reference model for later pruning and quantization experiments.

In [22]:
import time
import torch
from torch.utils.data import DataLoader

# Full CIFAR-10 training dataset
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

num_epochs = 1

mobilevit_train_losses = []
mobilevit_train_accuracies = []

start_time = time.time()

for epoch in range(num_epochs):
    mobilevit_model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        predicted = torch.argmax(logits, dim=1)
        total_samples += batch_size
        correct_predictions += (predicted == labels).sum().item()

        if batch_idx % 100 == 0:
            print(
                f"Epoch [{epoch + 1}/{num_epochs}], "
                f"Batch [{batch_idx + 1}/{len(train_loader)}], "
                f"Loss: {loss.item():.4f}"
            )

    epoch_loss = running_loss / total_samples
    epoch_accuracy = 100 * correct_predictions / total_samples

    mobilevit_train_losses.append(epoch_loss)
    mobilevit_train_accuracies.append(epoch_accuracy)

    print(
        f"\nEpoch [{epoch + 1}/{num_epochs}] completed "
        f"Loss: {epoch_loss:.4f} "
        f"Training Accuracy: {epoch_accuracy:.2f}%"
    )

end_time = time.time()
mobilevit_training_time = end_time - start_time

print(f"\nFull CIFAR-10 MobileViT Training Loss: {epoch_loss:.4f}")
print(f"Full CIFAR-10 MobileViT Training Accuracy: {epoch_accuracy:.2f}%")
print(f"Full CIFAR-10 MobileViT Training Time: {mobilevit_training_time:.2f} seconds")

Epoch [1/1], Batch [1/6250], Loss: 0.7059
Epoch [1/1], Batch [101/6250], Loss: 0.7669
Epoch [1/1], Batch [201/6250], Loss: 0.8987
Epoch [1/1], Batch [301/6250], Loss: 0.5809
Epoch [1/1], Batch [401/6250], Loss: 2.0130
Epoch [1/1], Batch [501/6250], Loss: 1.1241
Epoch [1/1], Batch [601/6250], Loss: 0.9607
Epoch [1/1], Batch [701/6250], Loss: 0.9982
Epoch [1/1], Batch [801/6250], Loss: 0.9225
Epoch [1/1], Batch [901/6250], Loss: 0.6335
Epoch [1/1], Batch [1001/6250], Loss: 1.0128
Epoch [1/1], Batch [1101/6250], Loss: 0.3516
Epoch [1/1], Batch [1201/6250], Loss: 0.8215
Epoch [1/1], Batch [1301/6250], Loss: 0.4048
Epoch [1/1], Batch [1401/6250], Loss: 0.4819
Epoch [1/1], Batch [1501/6250], Loss: 0.3987
Epoch [1/1], Batch [1601/6250], Loss: 0.9505
Epoch [1/1], Batch [1701/6250], Loss: 1.0519
Epoch [1/1], Batch [1801/6250], Loss: 1.3020
Epoch [1/1], Batch [1901/6250], Loss: 0.4331
Epoch [1/1], Batch [2001/6250], Loss: 0.7582
Epoch [1/1], Batch [2101/6250], Loss: 1.1022
Epoch [1/1], Batch [22

# Baseline MobileViT Evaluation — Full Test Dataset

The trained baseline MobileViT model is evaluated on the full CIFAR-10 test dataset to establish the reference classification accuracy before applying pruning and quantization.

In [23]:
import torch
from torch.utils.data import DataLoader

# Full CIFAR-10 test dataset
test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

mobilevit_model.eval()

correct_predictions = 0
total_samples = 0

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        predicted = torch.argmax(logits, dim=1)

        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        if batch_idx % 100 == 0:
            print(f"Evaluation batch [{batch_idx + 1}/{len(test_loader)}]")

mobilevit_test_accuracy = 100 * correct_predictions / total_samples

print(f"\nFull CIFAR-10 MobileViT Test Accuracy: {mobilevit_test_accuracy:.2f}%")

Evaluation batch [1/1250]
Evaluation batch [101/1250]
Evaluation batch [201/1250]
Evaluation batch [301/1250]
Evaluation batch [401/1250]
Evaluation batch [501/1250]
Evaluation batch [601/1250]
Evaluation batch [701/1250]
Evaluation batch [801/1250]
Evaluation batch [901/1250]
Evaluation batch [1001/1250]
Evaluation batch [1101/1250]
Evaluation batch [1201/1250]

Full CIFAR-10 MobileViT Test Accuracy: 76.75%


## Save Baseline MobileViT Model

In [24]:
torch.save(mobilevit_model.state_dict(), "baseline_mobilevit.pth")
print("Baseline MobileViT model saved.")

Baseline MobileViT model saved.


## Pruned MobileViT Model

In [25]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import copy

# Create a separate copy so the baseline model remains unchanged
pruned_mobilevit_model = copy.deepcopy(mobilevit_model)

# Apply 20% unstructured L1 pruning to Conv2d and Linear layers
for name, module in pruned_mobilevit_model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        prune.l1_unstructured(
            module,
            name="weight",
            amount=0.20
        )

# Make pruning permanent by removing pruning reparameterization
for name, module in pruned_mobilevit_model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        prune.remove(module, "weight")

pruned_mobilevit_model.to(device)

print("20% unstructured pruning applied to MobileViT model.")

20% unstructured pruning applied to MobileViT model.


## Evalutation of Pruned Model

In [26]:
import torch
from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

pruned_mobilevit_model.eval()

correct_predictions = 0
total_samples = 0

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = pruned_mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        predicted = torch.argmax(logits, dim=1)

        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        if batch_idx % 100 == 0:
            print(f"Evaluation batch [{batch_idx + 1}/{len(test_loader)}]")

pruned_mobilevit_test_accuracy = 100 * correct_predictions / total_samples

print(f"\nPruned MobileViT Test Accuracy: {pruned_mobilevit_test_accuracy:.2f}%")

Evaluation batch [1/1250]
Evaluation batch [101/1250]
Evaluation batch [201/1250]
Evaluation batch [301/1250]
Evaluation batch [401/1250]
Evaluation batch [501/1250]
Evaluation batch [601/1250]
Evaluation batch [701/1250]
Evaluation batch [801/1250]
Evaluation batch [901/1250]
Evaluation batch [1001/1250]
Evaluation batch [1101/1250]
Evaluation batch [1201/1250]

Pruned MobileViT Test Accuracy: 74.92%


## Quantized MobileViT Model

Dynamic post-training quantization is applied to the trained baseline MobileViT model. The quantized model is evaluated on the CIFAR-10 test set to measure the effect of reduced numerical precision on classification accuracy and inference time.

In [27]:
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Load a clean copy of the trained baseline model
quantized_mobilevit_model = copy.deepcopy(mobilevit_model)
quantized_mobilevit_model.load_state_dict(
    torch.load("baseline_mobilevit.pth", map_location="cpu")
)

# Dynamic quantization works on CPU and mainly affects Linear layers
quantized_mobilevit_model = torch.quantization.quantize_dynamic(
    quantized_mobilevit_model.cpu(),
    {nn.Linear},
    dtype=torch.qint8
)

quantized_mobilevit_model.eval()

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

correct_predictions = 0
total_samples = 0

start_time = time.time()

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.cpu()
        labels = labels.cpu()

        outputs = quantized_mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        predicted = torch.argmax(logits, dim=1)

        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        if batch_idx % 100 == 0:
            print(f"Evaluation batch [{batch_idx + 1}/{len(test_loader)}]")

end_time = time.time()

quantized_mobilevit_test_accuracy = 100 * correct_predictions / total_samples
quantized_mobilevit_inference_time = end_time - start_time

print(f"\nQuantized MobileViT Test Accuracy: {quantized_mobilevit_test_accuracy:.2f}%")
print(f"Quantized MobileViT Inference Time: {quantized_mobilevit_inference_time:.2f} seconds")

torch.save(
    quantized_mobilevit_model.state_dict(),
    "quantized_mobilevit_dynamic.pth"
)

print("Quantized MobileViT model saved.")

/tmp/ipykernel_42228/2838839157.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("baseline_mobilevit.pth", map_location="cpu")


Evaluation batch [1/1250]
Evaluation batch [101/1250]
Evaluation batch [201/1250]
Evaluation batch [301/1250]
Evaluation batch [401/1250]
Evaluation batch [501/1250]
Evaluation batch [601/1250]
Evaluation batch [701/1250]
Evaluation batch [801/1250]
Evaluation batch [901/1250]
Evaluation batch [1001/1250]
Evaluation batch [1101/1250]
Evaluation batch [1201/1250]

Quantized MobileViT Test Accuracy: 76.76%
Quantized MobileViT Inference Time: 65.76 seconds
Quantized MobileViT model saved.


## Evaluation of Quantized MobileViT Model

The dynamically quantized MobileViT model is evaluated on the full CIFAR-10 test dataset. Test accuracy and inference time are recorded to compare the quantized model against the baseline and pruned versions.

In [28]:
import time
import torch
from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

quantized_mobilevit_model.eval()

correct_predictions = 0
total_samples = 0

start_time = time.time()

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.cpu()
        labels = labels.cpu()

        outputs = quantized_mobilevit_model(images)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs

        predicted = torch.argmax(logits, dim=1)

        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        if batch_idx % 100 == 0:
            print(f"Evaluation batch [{batch_idx + 1}/{len(test_loader)}]")

end_time = time.time()

quantized_mobilevit_test_accuracy = 100 * correct_predictions / total_samples
quantized_mobilevit_inference_time = end_time - start_time

print(f"\nQuantized MobileViT Test Accuracy: {quantized_mobilevit_test_accuracy:.2f}%")
print(f"Quantized MobileViT Inference Time: {quantized_mobilevit_inference_time:.2f} seconds")

Evaluation batch [1/1250]
Evaluation batch [101/1250]
Evaluation batch [201/1250]
Evaluation batch [301/1250]
Evaluation batch [401/1250]
Evaluation batch [501/1250]
Evaluation batch [601/1250]
Evaluation batch [701/1250]
Evaluation batch [801/1250]
Evaluation batch [901/1250]
Evaluation batch [1001/1250]
Evaluation batch [1101/1250]
Evaluation batch [1201/1250]

Quantized MobileViT Test Accuracy: 76.76%
Quantized MobileViT Inference Time: 64.09 seconds


# Inference Time Comparison

This section compares the inference time of the baseline, pruned, and quantized MobileViT models to evaluate the computational efficiency gained through model compression.

In [33]:
import time
import torch
import pandas as pd
from torch.utils.data import DataLoader

# Use the same test loader for all models
inference_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

def measure_inference_time(model, data_loader, device_type="cpu", model_name="Model"):
    model.eval()

    correct_predictions = 0
    total_samples = 0

    start_time = time.time()

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(data_loader):
            if device_type == "cpu":
                images = images.cpu()
                labels = labels.cpu()
            else:
                images = images.to(device)
                labels = labels.to(device)

            outputs = model(images)
            logits = outputs.logits if hasattr(outputs, "logits") else outputs

            predicted = torch.argmax(logits, dim=1)

            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

            if batch_idx % 200 == 0:
                print(f"{model_name}: batch [{batch_idx + 1}/{len(data_loader)}]")

    end_time = time.time()

    accuracy = 100 * correct_predictions / total_samples
    inference_time = end_time - start_time
    average_time_per_image = inference_time / total_samples

    return accuracy, inference_time, average_time_per_image


baseline_accuracy, baseline_inference_time, baseline_time_per_image = measure_inference_time(
    mobilevit_model,
    inference_loader,
    device_type="cuda" if device.type == "cuda" else "cpu",
    model_name="Baseline MobileViT"
)

pruned_accuracy, pruned_inference_time, pruned_time_per_image = measure_inference_time(
    pruned_mobilevit_model,
    inference_loader,
    device_type="cuda" if device.type == "cuda" else "cpu",
    model_name="20% Pruned MobileViT"
)

quantized_accuracy, quantized_inference_time, quantized_time_per_image = measure_inference_time(
    quantized_mobilevit_model,
    inference_loader,
    device_type="cpu",
    model_name="Quantized MobileViT"
)

inference_results = pd.DataFrame({
    "Model": [
        "Baseline MobileViT",
        "20% Pruned MobileViT",
        "Quantized MobileViT"
    ],
    "Test Accuracy (%)": [
        baseline_accuracy,
        pruned_accuracy,
        quantized_accuracy
    ],
    "Total Inference Time (s)": [
        baseline_inference_time,
        pruned_inference_time,
        quantized_inference_time
    ],
    "Average Time per Image (s)": [
        baseline_time_per_image,
        pruned_time_per_image,
        quantized_time_per_image
    ]
})

inference_results = inference_results.round(4)

inference_results

Baseline MobileViT: batch [1/1250]
Baseline MobileViT: batch [201/1250]
Baseline MobileViT: batch [401/1250]
Baseline MobileViT: batch [601/1250]
Baseline MobileViT: batch [801/1250]
Baseline MobileViT: batch [1001/1250]
Baseline MobileViT: batch [1201/1250]
20% Pruned MobileViT: batch [1/1250]
20% Pruned MobileViT: batch [201/1250]
20% Pruned MobileViT: batch [401/1250]
20% Pruned MobileViT: batch [601/1250]
20% Pruned MobileViT: batch [801/1250]
20% Pruned MobileViT: batch [1001/1250]
20% Pruned MobileViT: batch [1201/1250]
Quantized MobileViT: batch [1/1250]
Quantized MobileViT: batch [201/1250]
Quantized MobileViT: batch [401/1250]
Quantized MobileViT: batch [601/1250]
Quantized MobileViT: batch [801/1250]
Quantized MobileViT: batch [1001/1250]
Quantized MobileViT: batch [1201/1250]


,Model,Test Accuracy (%),Total Inference Time (s),Average Time per Image (s)
0,Baseline MobileViT,76.75,56.0137,0.0056
1,20% Pruned MobileViT,74.92,55.8581,0.0056
2,Quantized MobileViT,76.76,64.7137,0.0065


## Final Model Comparison

The baseline, pruned, and quantized MobileViT models are compared using CIFAR-10 test accuracy and inference time where recorded.

In [35]:
import pandas as pd

final_results = pd.DataFrame({
    "Model": [
        "Baseline MobileViT",
        "20% Pruned MobileViT",
        "Quantized MobileViT"
    ],
    "Test Accuracy (%)": [
        baseline_accuracy,
        pruned_accuracy,
        quantized_accuracy
    ],
    "Accuracy Change vs Baseline (%)": [
        baseline_accuracy - baseline_accuracy,
        pruned_accuracy - baseline_accuracy,
        quantized_accuracy - baseline_accuracy
    ],
    "Inference Time (s)": [
        baseline_inference_time,
        pruned_inference_time,
        quantized_inference_time
    ],
    "Inference Time Difference from Baseline (s)": [
        baseline_inference_time - baseline_inference_time,
        pruned_inference_time - baseline_inference_time,
        quantized_inference_time - baseline_inference_time
    ],
    "Average Time per Image (s)": [
        baseline_time_per_image,
        pruned_time_per_image,
        quantized_time_per_image
    ]
})

final_results = final_results.round(4)

final_results

,Model,Test Accuracy (%),Accuracy Change vs Baseline (%),Inference Time (s),Inference Time Difference from Baseline (s),Average Time per Image (s)
0,Baseline MobileViT,76.75,0.00,56.0137,0.0000,0.0056
1,20% Pruned MobileViT,74.92,-1.83,55.8581,-0.1556,0.0056
2,Quantized MobileViT,76.76,0.01,64.7137,8.7000,0.0065


The baseline MobileViT model achieved 76.75% test accuracy. The 20% pruned model produced a small accuracy reduction, reaching 74.92%, while having a very similar inference time to the baseline. The dynamically quantized model achieved comparable accuracy to the baseline, but did not improve inference speed in this experiment. This suggests that dynamic quantization of only the linear layers may be insufficient to accelerate MobileViT on this CPU setup.